In [3]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import time

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
from keras.losses import CategoricalCrossentropy
import csv

from loss import *
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator as DAE_OptDG
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
from plotting import *


In [1]:
import models
import importlib
importlib.reload(models)
from models import *

thresholds = [1,240,800]
weights_path = './SQ_DAE/model-bb57b09d-SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST-checkpoints'
model = CreateSQAE_MaxTransformerReg_fromweights(weights_path,
                             input_shape=(16,16,2),   
                             patch_size=(3,4),        
                             embed_dim=64,           
                             num_heads=4,            
                             ff_dim=128,              
                             num_layers=4,            
                             dropout=0.1,
                             head_dims=(256,128,64),
                             threshold_offset=0,
                             n_bits=2,
                             initial_thresholds=thresholds,
                             final_outputs=3         
                            )
model.compile(
        optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
        loss=custom_sse_loss
    )


model.summary()

2026-09-03 04:11:03.134965: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading model from ./SQ_DAE/model-bb57b09d-SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.754-t95021.8672-v101612.5859.hdf5


2026-09-03 04:11:13.420759: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-09-03 04:11:13.474092: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-09-03 04:11:13.475547: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-09-03 04:11:13.476442: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

layer_normalization_8
(None, 20, 64)


NameError: name 'custom_sse_loss' is not defined

In [17]:
import importlib
import DAE_DataGenerator
import dataset_utils
import SoftQuantizeLayer
import AnnealingScheduler

importlib.reload(DAE_DataGenerator)
importlib.reload(dataset_utils)
importlib.reload(SoftQuantizeLayer)
importlib.reload(AnnealingScheduler)

from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
#from dataprep import *
# from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *

In [7]:
dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "tfrecords")

# dataset_dir_train = os.path.join(dataset_base_dir, "/dataset_3sr_16x16_50x12P5_parquets", 'train/')
# dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets", 'test/')

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

tfrecords_dir_train   = os.path.join(tfrecords_base_dir, "TFR_train",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')


tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3srCI_16x16_0eN_raw')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3srCI_16x16_0eN_raw')



print(dataset_dir_val)
batch_size = 5000
val_batch_size = 5000
train_file_size = 80
val_file_size = 20

#### Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=False,
    # seed=42,
    quantize=False,
    # bitwise_categorical = True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=False,
    # seed=42,
    quantize=False,
    # bitwise_categorical = True,
)

/uscms/home/bweiss/nobackup/smart-pixels/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/test_contained/
Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_val/3srCI_16x16_0eN_raw/metadata.json


Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_train/3srCI_16x16_0eN_raw/metadata.json


In [17]:
from datetime import datetime
tname = 'SQDAE_SlimTransformer_Regression_2bit_80eN_Target_0_30_845_TEST'#0_7_23_134_404_867_1748'
fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./SQ_DAE/model-{fingerprint}-{tname}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.4f}-v{val_loss:.4f}.hdf5'
print("fingerprint: ", fingerprint)

fingerprint:  35ec9d9c


In [18]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

early_stopping_patience = 1000

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort(key=lambda x: os.path.getmtime(os.path.join(base_dir,x)))
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=400.0, 
    verbose=1      
)

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)


In [19]:
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=5000,
        shuffle=False,
        verbose=1
    )

Epoch 1/5000
 1/84 [..............................] - ETA: 4:07 - loss: 2158.7588

2026-08-26 14:24:47.592630: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


84/84 [==============================] - ETA: 0s - loss: 311.2565
Epoch 1: val_loss improved from inf to 72.32120, saving model to ./SQ_DAE/model-35ec9d9c-SQDAE_SlimTransformer_Regression_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.01-t311.2565-v72.3212.hdf5
84/84 [==============================] - 35s 391ms/step - loss: 311.2565 - val_loss: 72.3212
Epoch 2/5000
84/84 [==============================] - ETA: 0s - loss: 89.5637
Epoch 2: val_loss improved from 72.32120 to 46.86477, saving model to ./SQ_DAE/model-35ec9d9c-SQDAE_SlimTransformer_Regression_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.02-t89.5637-v46.8648.hdf5
84/84 [==============================] - 8s 97ms/step - loss: 89.5637 - val_loss: 46.8648
Epoch 3/5000
84/84 [==============================] - ETA: 0s - loss: 70.6085
Epoch 3: val_loss improved from 46.86477 to 38.03767, saving model to ./SQ_DAE/model-35ec9d9c-SQDAE_SlimTransformer_Regression_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.03-t70.6085-v

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



84/84 [==============================] - ETA: 0s - loss: 10.9140
Epoch 622: val_loss did not improve from 11.34164
84/84 [==============================] - 6s 74ms/step - loss: 10.9140 - val_loss: 11.4232
Epoch 623/5000
83/84 [============================>.] - ETA: 0s - loss: 11.0600
Epoch 623: val_loss did not improve from 11.34164
84/84 [==============================] - 6s 68ms/step - loss: 11.0580 - val_loss: 11.3995
Epoch 624/5000
84/84 [==============================] - ETA: 0s - loss: 10.9278
Epoch 624: val_loss did not improve from 11.34164
84/84 [==============================] - 6s 69ms/step - loss: 10.9278 - val_loss: 11.3843
Epoch 625/5000
84/84 [==============================] - ETA: 0s - loss: 10.8812
Epoch 625: val_loss did not improve from 11.34164
84/84 [==============================] - 6s 73ms/step - loss: 10.8812 - val_loss: 11.4183
Epoch 626/5000
84/84 [==============================] - ETA: 0s - loss: 11.0163
Epoch 626: val_loss did not improve from 11.34164
84/84